### Templates

**- start from end (know future element before knowing current) - i < j < k**

**- use stack : when 2 dependent for loop present**

```bash
"""
# Pattern: Reduce O(n³) to O(n²) for problems with constraint i < j < k
# Strategy: Fix middle element j, iterate k forward, track i backward using stack
"""
# Phase 1: Forward pass - iterate j and k
for j in range(0,n-1):
    for k in range(j+1,n):
        # Calculate result combining a[j], a[k], and preprocessed info about i (i < j)
        # Use previously computed data about elements before j
    # track/update state for position j

# Phase 2: Backward pass - preprocess data about "previous" elements
for i in range(n-1,-1,-1): # Start from end to know "future" before "current"
    # Calculate result: combine a[i] with elements after it (stored in stack)
    
    while len(stk)>0 and stk[-1]< a[i]: # Maintain ascending stack (pop smaller)
        # stk[-1] > a[i] would maintain descending stack (pop larger)
        j = stk.pop()  # j is a candidate but not the largest/best
        # Process: a[i] is now the "answer" for this j
    stk.append(a[i]) # Push current element for future iterations
```

## 901. Online Stock Span

Input - [100,80,60,70,60,75,85]
Output - for current number count how many numbers are less than or equal to current number

Output - Interactive mode (user will keep on adding in stocks array & keep calling the below function for current stock span)
```bash
100 -> 1 [100]
80 -> 1 [80]
60 -> 1 [60]
70 -> 2 [70,60]
60 -> 1 [60] # continuous
75 -> 4 [75,60,70,60]
85 -> 6 [85,75,60,70,60,80]
```
```python
class StockSpanner:
    def __init__(self):
        self.stack = []
    def next(self, price: int) -> int:
        span = 1
        while len(self.stack)>0 and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append([price,span])
        return span

# Your StockSpanner object will be instantiated and called as such:
# obj = StockSpanner()
# param_1 = obj.next(price)

```

## Stack using Queue

```python

# push() → O(1)
# pop() → amortized O(1)
# peek() → O(1) using peekEl
# empty() → O(1)

class MyQueue:
    def __init__(self):
        self.input = []     # stack 1
        self.output = []    # stack 2
        self.peekEl = -1

    def push(self, x: int) -> None:
        if not self.input:
            self.peekEl = x   # store the next front element
        self.input.append(x)  # push onto input stack
    # Amortized O(1)
    def pop(self) -> int:
        # If output is empty, transfer from input → output
        if not self.output:
            while self.input:
                self.output.append(self.input.pop())
        return self.output.pop()
    def peek(self) -> int:
        # If output is empty, front element is peekEl
        if not self.output:
            return self.peekEl
        return self.output[-1]
    # empty - when both are empty
    def empty(self) -> bool:
        return not self.input and not self.output
```

## 1047. Remove All Adjacent Duplicates In String

A duplicate removal consists of choosing two adjacent and equal letters and removing them.
We repeatedly make duplicate removals on s until we no longer can. The output will be unique
```bash
"abbaca" -> "aaca" ("bb" removed) -> "ca" ("aa" removed)
"azxxzy" -> "azzy" -> "ay"
```
```python
def removeDuplicates(self, s: str) -> str:
    def process(index: int) -> list:
        if index == len(s): # end of string
            return []
        stack = process(index + 1)
        # Logic: Compare current character with top of the stack
        if not stack or stack[-1] != s[index]:
            stack.append(s[index])
        else:
            stack.pop()
        return stack

    # Start recursion from index 0
    result_list = process(0)
    return "".join(result_list[::-1]) # stack store in reverse order
```

## 224. Basic Calculator

Input - "10 - (4+5+2) -3 + (6+8)"

```bash
1. digit  => build the number digit by digit
2. + or - => the number you are building got finalized, 
            add to result, reset number = 0
3. "("  => save them in stack,reset result=0,start calculating
            sum inside the parentheses from scratch
4. ")"  => finish the sum inside parentheses, pull old sign & 
            result from stack multiply the internal sum by the sign that 
            preceded the parenthesis, then add that to the result that
            existed before the parenthesis
    
## Edge case
[2,-,1,+,2] -> evaluated = 2-1 + 2 = 1+2 , here last 2 will be stored in number & not added in result since no "+" after 2, 
result += number*sign # at the end of loop
```

```python
def calculate(self, s: str) -> int:
    stack = []
    number = 0
    result = 0
    sign = 1
    
    for i in range(len(s)):
        if s[i].isdigit():
            number = 10*number + (ord(s[i]) - ord('0'))
        elif s[i]=="+":
            result += number*sign
            sign = 1
            number = 0
        elif s[i]=="-":
            result += number*sign
            sign =-1
            number = 0
        elif s[i]=="(":
            stack.append(result)
            stack.append(sign)
            result = 0
            number = 0
            sign = 1
        elif s[i]==")":
            # Finalize the number currently being built inside the 
            # parentheses
            result += number*sign
            number = 0
            # Multiply by the sign that was active BEFORE the '('
            # The stack.pop() here retrieves the 'sign' we saved 
            # when we saw '('
            result = result * stack.pop()
            # Add the result that was active BEFORE the '('
            # The stack.pop() here retrieves the 'result' we 
            # saved when we saw '('
            result += stack.pop()
    result += number*sign
    return result

```

## 907. Sum of Subarray Minimums

Given an array of integers arr, find the sum of min(b), where b ranges over every (contiguous) subarray of arr.Since the answer may be large, return the answer modulo 10^9 + 7.

```bash
arr = [3,1,2,4]
Output: 17
Subarrays are [3], [1], [2], [4], [3,1], [1,2], [2,4], [3,1,2], [1,2,4], [3,1,2,4]. 
Minimums are 3, 1, 2, 4, 1, 1, 2, 1, 1, 1.
Sum of Minimums is 17.
```

```python
## Brute force - all subarray (n^2), min calculation (n)
for i in range(n):
    minV = arr[i]
    for j in range(i,n):
        minV = min(minV,arr[j])
        result =(result + minV)% M

"""
Why stack ?

i - NSL[i] =  count of how many starting positions exist for a subarray such that arr[i] is the minimum (moving left).

arr = [3, 1, 2, 4]

Consider element 2 at index 2:

- NSL is index 1 (value 1 is the nearest smaller to the left).
- NSR is index 4 (boundary is the end of array, as no smaller element exists to the right).
- left = 2 - 1 = 1, right = 4 - 2 = 2
- Subarrays where 2 is the minimum are: [2] and [2, 4] 
  i.e 1 * 2 = 2 subarrays
- Contribution to total: 2 * 2 = 4

"""

def sumSubarrayMins(self, arr: List[int]) -> int:
    n = len(arr)
    # NSL : next smaller to left
    def get_NSL():
        NSL = [0 for _ in range(n)]
        stack = []
        for i in range(n):
            if len(stack)==0:
                NSL[i]=-1
            else:
                while len(stack)>0 and arr[stack[-1]]>arr[i]:
                    stack.pop()
                NSL[i] = -1 if len(stack)==0 else stack[-1]
            stack.append(i)
        return NSL
    # NSR : next smaller to right
    def get_NSR():
        NSR = [0 for _ in range(n)]
        stack = []
        for i in range(n-1,-1,-1):
            if len(stack)==0:
                NSR[i] = n
            else:
                while len(stack)>0 and arr[stack[-1]]>=arr[i]:
                    stack.pop()
                NSR[i] = n if len(stack)==0 else stack[-1]
            stack.append(i)
        return NSR

    NSL = get_NSL()
    NSR = get_NSR()
    M = 1e9+7
    sums = 0
    for i in range(n):
        left = i - NSL[i]
        right = NSR[i] - i
        combined = (left*right)%M
        sums = (sums + (arr[i]*combined)%M)%M
    return int(sums)

```


## 739. Daily Temperatures

Find answer[i] = number of days you have to wait after the i-th day to get a warmer temperature, if there is no future day for which this is possible, keep answer[i] == 0

    Input: temperatures = [73,74,75,71,69,72,76,73]
    Output: [1,1,4,2,1,1,0,0]
    
```python

def get_NGR():
    NGR = [-1 for _ in range(n)]
    stack = []
    for i in range(n-1,-1,-1):
        if len(stack)==0:
            NGR[i]=0
        while len(stack)>0 and arr[stack[-1]]<arr[i]: # strictly decreasing from top → bottom
            stack.pop()
        NGR[i] = 0 if len(stack)==0 else stack[-1]-i # we need days,not index
        stack.append(i)
    return NGR

```

## 71. Simplify Path

You are given an absolute path which always begins with a slash '/'. Your task is to transform this absolute path into its simplified canonical path.

The rules of a Unix-style file system are as follows:

    '.' -> current directory.
    '..' ->  previous/parent directory.
    Multiple slashes such as '//' and '///' are treated as a single slash '/'.


The simplified canonical path should follow these rules:

    The path must start with a single slash '/'.
    Directories within the path must be separated by exactly one slash '/'.
    The path must not end with a slash '/', unless it is the root directory.
    The path must not have any single or double periods ('.' and '..') used to denote current or parent directories.
Return the simplified canonical path.

```bash
Input - "/a/./b/../..//c/" 
Output - / -> a -> b -> a -> / -> c

tokenized/split on "/" -> [ "", "a", ".", "b", "..", "..", "", "c", "" ]



1. Ignore "" and ".":
   Empty strings occur when there are multiple slashes (e.g., //)

2. Handle `".." (Parent Directory):
   - If we see .., we want to move up. In code, this means stack.pop().
   - Safety Check: if len(stack) > 0 ensures we don't try to go 
     above the root directory (you can't go up from /).
3. Building result 
   - uses result = "/" + stack.pop() + result to prepend 
   the directories. This correctly maintains the left-to-right 
   order of the path.
```

```python
def simplifyPath(self, path: str) -> str:
        split_paths = path.split("/")
        stack = []
        for i in range(len(split_paths)):
            if split_paths[i]=="" or split_paths[i]==".":
                continue
            # folder name - add to stack
            if split_paths[i]!="..":
                stack.append(split_paths[i])
            elif len(stack)>0: # we got "..",go to parent
                stack.pop()
                        
        result = ""
        # root
        if len(stack)==0:
            result = "/"
        else:
            while len(stack)>0:
                result = "/"+stack.pop()+result
        
        return result

```

## 946. Validate Stack Sequences

Given two integer arrays pushed and popped each with distinct values, return true if this could have been the result of a sequence of push and pop operations on an initially empty stack, or false otherwise.

```code

pushed = [1,2,3,4,5], popped = [4,5,3,2,1]

i=0, j=0
stack = [1]

i=1,j=0
stack = [1,2]

i=2,j=0
stack = [1,2,3]

i=3,j=0
stack = [1,2,3,4] 
stack.top==popped[j=0] -> stack.pop
stack = [1,2,3]
j = 1

i=4,j=1
stack = [1,2,3,5]
stack.top==popped[j=1] -> stack.pop
stack = [1,2,3]
j=2

i=4,j=2
stack = [1,2,3]
stack.top==popped[j=2] -> stack.pop
stack = [1,2]
j=3

i=4,j=3
stack = [1,2]
stack.top==popped[j=3] -> stack.pop
stack = [1]

i=4,j=4  # end
stack = [1]
stack.top==popped[j=4] -> stack.pop
stack = [] # True

```

```python
i, j = 0,0
while i < n and j < n:
        stack.append(pushed[i])
        while len(stack)>0 and stack[-1]==popped[j]:
        stack.pop()
        j+=1
        i+=1

return len(stack)==0

```

## 735. Asteroid Collision

We are given an array asteroids of integers representing asteroids in a row. The indices of the asteroid in the array represent their relative position in space.

For each asteroid, the absolute value represents its size, and the sign represents its direction (positive meaning right, negative meaning left). Each asteroid moves at the same speed.

Find out the state of the asteroids after all collisions. If two asteroids meet, the smaller one will explode. If both are the same size, both will explode. Two asteroids moving in the same direction will never meet.


        Example 1:

        Input: asteroids = [5,10,-5]
        Output: [5,10]
        Explanation: The 10 and -5 collide resulting in 10. The 5 and 10 never collide.
        Example 2:

        Input: asteroids = [8,-8]
        Output: []
        Explanation: The 8 and -8 collide exploding each other.
        Example 3:

        Input: asteroids = [10,2,-5]
        Output: [10]
        Explanation: The 2 and -5 collide resulting in -5. The 10 and -5 collide resulting in 10.
        Example 4:

        Input: asteroids = [3,5,-6,2,-1,4]​​​​​​​
        Output: [-6,2,4]
        Explanation: The asteroid -6 makes the asteroid 3 and 5 explode, and then continues going left. On the other side, the asteroid 2 makes the asteroid -1 explode and then continues going right, without reaching asteroid 4.

```python

# collision will happen if asteroid < 0 and stack.top > 0
1. run loop take each asteroid 
2. while not stack.empty and current asteroid < 0 and stack.top > 0
    2.1 calculate result # don't pop from stack yet
    # current astroid < 0 & it'll destroy more 
    2.2 result < 0 : so pop form stack 
    2.3 result > 0 : no more poping, break out of loop & destory current astroid = 0
    2.4 result == 0 : pop form stack ,destory current astroid = 0
# non-zero remaining after collision
3. push astoroid to stack

# remaining ones are the result after collision
4. once all astoroids covered -> pop all of them & store in result 

```

## 456. 132 Pattern

Given an array of n integers nums, a 132 pattern is a subsequence of three integers nums[i], nums[j] and nums[k] such that i < j < k and nums[i] < nums[k] < nums[j].

Return true if there is a 132 pattern in nums, otherwise, return false.

**Approach -1**

**- convert 3 for loops to 2 for loops : use a variable & update it after 2nd loop**
```cpp
for(int i = 0; i<n; i++) {
    for(int j = i+1; j<n; j++) {
        if(nums[j] > nums[i]) {
           for(int k = j+1; k<n; k++) {
                if(nums[i] < nums[k] && nums[k] < nums[j] )
                        return true;
            }     
        }
    }
}

```


```code 

        0 1 2 3 
nums = [3,1,4,2]

j = 0
-------------------
k = 1, min_i = nums[j=0] = 3
k = 2, min_i=3 < nums[k=2]=4 > nums[j=0]=3
k = 3, min_i=3 > nums[k=3]=2

j = 1
-------------------
min_i = min(3,nums[j=1]) = 1
k = 2, min_i = 1 < nums[k=2]=4 > nums[j=1]=1
k = 3, min_i = 1 < nums[k=3]=2 > nums[j=1]=1

j=2
-----------------------
min_i = 1
k = 3, min_i = 1 < nums[k=3]=2 < nums[j=2]=4
```
**Approach - 2**

**- start from end (know future element before knowing current) - i < j < k**
**- use stack : when 2 dependent for loop present**

```code
         0 1 2 3
nums = [-1,3,2,0]

i = 3
-------------------
stk = [0]

i = 2
----------------
nums_k = 0 , since nums[i=2] = 2 > stk[-1]= 0
stk = [2]

i = 1
--------------
nums[i=1]=3 > nums_k=0 
num_k = 2, since nums[i=1] = 3 > stk[-1]=2
stk = [3]

i = 0
------------------
nums[i=0]=-1 < nums_k=2
found = [-1,3,2]

nums = [3,5,0,3,4]

num1 < num3 < num2
                   stk = [4]
                   stk = [4,3]
                   stk = [4,3,0] , stored in descending order 
     0 -> 3 
     3 -> 4     5  stk = [5]  
3    4          5 , when we get nums[i] < num3 -> assign to num1,found seq

```

```python
## Approach -1 O(n^2)

def find132pattern(self, nums: List[int]) -> bool:
    n = len(nums)
    if n < 3: return False
    min_i = float('inf')
    for j in range(0,n-1):
        for k in range(j+1,n):
            if min_i < nums[k] < nums[j]:
                return True
        min_i = min(min_i,nums[j])
    return False

## Approach - 2 O(n)

def find132pattern(self, nums: List[int]) -> bool:
        n = len(nums)
        if n < 3: return False
        stk = []
        num_k = float('-inf')
        for i in range(n-1,-1,-1):
            if nums[i]<num_k:
                return True
            while len(stk)>0 and stk[-1]<nums[i]:
                num_k = stk.pop()
            stk.append(nums[i])

        return False

```